# Advanced Method: Generative Modern Hopfield Sampler for Stylized Market Facts

This notebook records the final method used in `L6d` to generate synthetic market returns with a modern Hopfield model.

The focus here is method derivation and interpretation, not implementation details. For executable code, see [L6d Lab Solution](CHEME-5820-L6d-Lab-Solution-Spring-2026.ipynb).

> __Learning Objectives:__
>
> By the end of this notebook, you should be able to:
>
> * __Reconstruct the generator pipeline:__ Identify each state variable in the synthetic-day loop and explain how one new return vector is produced. Track how the query state, probability vector, and generated return vector evolve from day $t$ to day $t+1$.
> * __Explain why the sampler can match key stylized facts:__ Distinguish which parts of the method influence tail behavior, linear autocorrelation, and volatility clustering. Connect each empirical behavior to a specific mechanism in the sampling equations.
> * __Tune the model with controlled trade-offs:__ Describe how persistence and noise parameters change smoothness, clustering strength, and realism. Use a parameter-first view to decide when to favor shape fidelity versus moment matching.

Let's get started!

___


## Method Construction

We model each trading day as a feature vector over firms, and generate a synthetic path by iterating a stateful Hopfield-driven map.

> __Notation and Objects:__
>
> Let $F$ be the number of firms and $K$ the number of historical memory days. The raw memory matrix is $oldsymbol{M}\\in\\mathbb{R}^{F\\times K}$ with memory day vectors in columns.
>
> Let $\\boldsymbol{s}_{t}\\in\\mathbb{R}^{F}$ denote the query state at synthetic day $t$. Let $\\boldsymbol{p}_{t}\\in\\Delta^{K-1}$ denote the final modern-Hopfield memory probability vector returned by `recover(...)`, where $\\Delta^{K-1}$ is the probability simplex.

For each day $t$, the generator applies the following sequence:

1. __Recover memory probabilities from current query__
$$
\\boldsymbol{p}_{t} = \\mathrm{RecoverProbabilities}(\\boldsymbol{s}_{t};\\,\\beta,\\epsilon_{p},\\epsilon_{s},\\mathrm{maxiterations}).
$$

2. __Form stochastic top-$k$ proposal__
$$
\\tilde{p}_{t,j} = p_{t,j} + \\sigma_{\\mathrm{select}}\\,\\varepsilon_{t,j},\\qquad \\varepsilon_{t,j}\\sim\\mathcal{N}(0,1),
$$
sort by $\\tilde{p}_{t,j}$, keep top-$k$ indices $\\mathcal{I}_{t}$, and define:
$$
q^{\\mathrm{new}}_{t,j} =
\\begin{cases}
p_{t,j}, & j\\in\\mathcal{I}_{t},\\
0, & j\\notin\\mathcal{I}_{t},
\\end{cases}
\\qquad
\\boldsymbol{q}^{\\mathrm{new}}_{t} \\leftarrow \\frac{\\boldsymbol{q}^{\\mathrm{new}}_{t}}{\\sum_{j=1}^{K} q^{\\mathrm{new}}_{t,j} + 10^{-12}}.
$$

3. __Sticky mixture weights and sharpening__
$$
\\boldsymbol{q}_{t} = \\lambda_{q}\\,\\boldsymbol{q}_{t-1} + (1-\\lambda_{q})\\,\\boldsymbol{q}^{\\mathrm{new}}_{t},
\\qquad
\\boldsymbol{q}_{t} \\leftarrow \\frac{\\boldsymbol{q}_{t}}{\\sum_{j=1}^{K} q_{t,j} + 10^{-12}},
$$
$$
q_{t,j} \\leftarrow q_{t,j}^{\\gamma_{\\mathrm{sharpen}}},
\\qquad
\\boldsymbol{q}_{t} \\leftarrow \\frac{\\boldsymbol{q}_{t}}{\\sum_{j=1}^{K} q_{t,j} + 10^{-12}}.
$$

4. __Base generated vector and latent volatility modulation__
$$
\\boldsymbol{m}^{\\mathrm{raw}}_{t} = \\boldsymbol{M}\\boldsymbol{q}_{t}.
$$
Let $\\boldsymbol{\\mu}_{\\mathrm{orig}}\\in\\mathbb{R}^{F}$ be the per-ticker sample mean of historical returns. Define a latent scalar log-volatility state:
$$
\\log h_{t} = \\rho_{\\log h}\\,\\log h_{t-1} + \\sigma_{\\log h}\\,\\xi_{t},\\qquad \\xi_{t}\\sim\\mathcal{N}(0,1),
$$
$$
h_{t}=\\exp(\\log h_{t}),\\qquad
\\boldsymbol{m}_{t} = \\boldsymbol{\\mu}_{\\mathrm{orig}} + \\sqrt{h_{t}}\\left(\\boldsymbol{m}^{\\mathrm{raw}}_{t}-\\boldsymbol{\\mu}_{\\mathrm{orig}}\\right).
$$

5. __Query-state evolution__
$$
\\boldsymbol{s}_{t+1} = \\lambda_{\\mathrm{query}}\\,\\boldsymbol{s}_{t} + (1-\\lambda_{\\mathrm{query}})\\,\\boldsymbol{m}_{t} + \\sigma_{\\mathrm{query}}\\,\\boldsymbol{\\eta}_{t},\\qquad \\boldsymbol{\\eta}_{t}\\sim\\mathcal{N}(\\boldsymbol{0},\\boldsymbol{I}_{F}).
$$

6. __Optional post-adjustments (moment matching)__

Let $\\boldsymbol{\\mu}_{\\mathrm{gen}}$ and $\\boldsymbol{\\sigma}_{\\mathrm{gen}}$ be generated per-ticker means and standard deviations, and $\\boldsymbol{\\sigma}_{\\mathrm{orig}}$ historical per-ticker standard deviations.

If mean matching is enabled:
$$
\\widehat{\\boldsymbol{G}} \\leftarrow \\widehat{\\boldsymbol{G}} + \\boldsymbol{1}\\left(\\boldsymbol{\\mu}_{\\mathrm{orig}}-\\boldsymbol{\\mu}_{\\mathrm{gen}}\\right)^{\\top}.
$$

If standard-deviation matching is enabled with partial scaling exponent $\\alpha_{\\mathrm{std}}\\in[0,1]$:
$$
\\boldsymbol{a} = \\left(\\frac{\\boldsymbol{\\sigma}_{\\mathrm{orig}}}{\\boldsymbol{\\sigma}_{\\mathrm{gen}} + 10^{-12}}\\right)^{\\alpha_{\\mathrm{std}}},
$$
$$
\\widehat{\\boldsymbol{G}} \\leftarrow \\left(\\widehat{\\boldsymbol{G}} - \\boldsymbol{1}\\boldsymbol{\\mu}_{\\mathrm{gen}}^{\\top}\\right)\\operatorname{diag}(\\boldsymbol{a}) + \\boldsymbol{1}\\boldsymbol{\\mu}_{\\mathrm{orig}}^{\\top}.
$$


## Stylized-Facts Interpretation

This section links each stylized fact to a specific part of the sampler.

> __Why each behavior appears:__
>
> __Fat tails:__ The generated day is a nonlinear map of top-$k$ memory mixtures with stochastic selection and volatility modulation. This creates heavier tails than a single Gaussian linear model because both state switching and log-vol shocks inject regime-like variability.
>
> __Low linear autocorrelation in returns:__ Setting $\\lambda_{q}\\approx 0$ and $\\lambda_{\\mathrm{query}}\\approx 0$ suppresses direct carry-over in the return level. The resulting process can keep $\\operatorname{Corr}(r_{t}, r_{t-1})$ near zero while still remaining non-Gaussian.
>
> __Volatility clustering:__ Persistence in $\\log h_{t}$ induces persistence in return magnitudes without requiring persistence in return signs. This targets positive autocorrelation in $|r_{t}|$ or $r_{t}^{2}$, which is the standard diagnostic for clustering.

For diagnostics, compute both level and magnitude autocorrelation:
$$
\\rho_{1}^{(r)} = \\operatorname{Corr}(r_{t}, r_{t-1}),\\qquad
\\rho_{1}^{(|r|)} = \\operatorname{Corr}(|r_{t}|, |r_{t-1}|),\\qquad
\\rho_{1}^{(r^{2})} = \\operatorname{Corr}(r_{t}^{2}, r_{t-1}^{2}).
$$

A good target profile is $\\rho_{1}^{(r)}\\approx 0$ with $\\rho_{1}^{(|r|)}>0$ and $\\rho_{1}^{(r^{2})}>0$.


## Practical Tuning Guide

Use the following controls to tune realism in a stable way:

* __Volatility clustering strength (`\\rho_logh`, `\\sigma_logh`):__ Increase `\\rho_logh` to lengthen volatility persistence in time. Increase `\\sigma_logh` to widen high-volatility episodes and strengthen clustering diagnostics on $|r_t|$ and $r_t^2$.
* __Raw-return memory (`\\lambda_q`, `\\lambda_query`):__ Keep both near zero when you want weak level autocorrelation. Increase either one only if you explicitly want stronger return-level persistence.
* __Mixture smoothness (`k`, `\\gamma_sharpen`):__ Larger `k` smooths more and can narrow distributions. Larger `\\gamma_sharpen` concentrates mass and tends to increase dispersion and tail weight.
* __Moment correction (`match_ticker_means`, `match_ticker_stds`, `\\alpha_std`):__ Mean matching is usually safe and improves calibration stability. Standard-deviation matching should remain optional because aggressive scaling can mask intrinsic clustering structure.

A practical workflow is: tune dynamic parameters first to match autocorrelation signatures, then apply light moment correction only if needed.


## Summary

This method builds synthetic market returns by combining modern Hopfield memory probabilities with a stateful sampling process and a persistent latent volatility factor.

The key implication is that stylized facts are controlled by separate parameter blocks, so calibration can be done in a structured way instead of by trial-and-error edits.

> __Key takeaways:__
>
> * __Stateful sampling is necessary for clustering:__ A memoryless daily sampler can reproduce fat tails and weak linear autocorrelation but usually misses persistent volatility regimes. Introducing a persistent latent log-vol state provides a direct mechanism for clustered magnitudes without forcing return-level autocorrelation.
> * __Autocorrelation targets require separate diagnostics:__ You should evaluate both return autocorrelation and magnitude autocorrelation to assess realism. Matching only $\\operatorname{Corr}(r_t,r_{t-1})$ is insufficient because clustering lives in $|r_t|$ and $r_t^2$.
> * __Moment matching is a calibration layer, not a generator mechanism:__ Mean and standard-deviation corrections can improve cross-sectional calibration after synthesis. They should be applied carefully because they do not create clustering and can alter visual distribution shape if overused.

___


### References
1. Hopfield JJ (1982), *Neural networks and physical systems with emergent collective computational abilities*, PNAS 79(8):2554-2558. DOI: https://doi.org/10.1073/pnas.79.8.2554
2. Ramsauer H, Schafl B, Lehner J, et al. (2021), *Hopfield Networks is All You Need*, ICLR 2021. arXiv: https://arxiv.org/abs/2008.02217
3. Cont R (2001), *Empirical properties of asset returns: stylized facts and statistical issues*, Quantitative Finance 1(2):223-236. DOI: https://doi.org/10.1080/713665670

Note: the generator equations in this notebook are method-specific to the L6d workflow and are documented here to make the calibration logic reproducible.
